# Task 1.3 — What the Paper Claims to Improve
**Paper**: *Efficient Online Learning for Large-Scale Sparse Kernel Logistic Regression* (AAAI 2012)

## Main Baseline / Prior Method

The primary baseline the paper positions itself against is the **Non-Conservative online learning algorithm (NC)** for Kernel Logistic Regression (Algorithm 1 in the paper). NC is simply stochastic gradient descent applied to the constrained KLR formulation in Eq. (2). Additionally, the paper compares against **batch-mode coordinate descent methods** — Primal Coordinate Descent (PCD) and Dual Coordinate Descent (DCD) — based on Keerthi et al. (2005), as well as the **Import Vector Machine (IVM)** by Zhu and Hastie (2001). Among online methods, the secondary comparison is with **Pegasos** (Shalev-Shwartz, Singer, and Srebro 2007), the state-of-the-art online kernel SVM.

## Limitation of the Baseline Identified by the Paper

The NC algorithm updates the kernel classifier on *every* received training example. Because the logistic loss $\ell(z) = \ln(1 + e^{-z})$ is strictly positive for all finite $z$, the gradient is never exactly zero, so every example contributes a non-zero weight to the model. The result is a completely **dense** classifier: after seeing $T$ examples, the model has $T$ support vectors. This makes both training (kernel matrix row computation per step) and testing (summing $T$ kernel evaluations for each prediction) expensive — exactly the opposite of what you want for large-scale deployment. The batch-mode methods (PCD, DCD) face the same density issue and additionally require storing and manipulating the full kernel matrix, while IVM's greedy selection has $O(n^2 q^2)$ cost that is impractical on large datasets.

## How the Proposed Method Overcomes This Limitation

The conservative algorithms (Margin and Auxiliary) introduce a **Bernoulli gating mechanism** before each update. A sampling probability $p_t$ is computed based on how confidently the current classifier already handles the incoming example — if the model is already correct and confident, $p_t$ is low, and the example is likely skipped. This way, only the "informative" examples end up as support vectors, shrinking the classifier. The Auxiliary variant further decouples the sparsity control from the learning rate by introducing a tuneable parameter $\gamma$.

## Scenario Where the Proposed Method Would Not Outperform the Baseline

When the classification problem is inherently difficult — for instance, a dataset with **high label noise** or where the classes overlap heavily in kernel space — most training examples will have low posterior confidence $p(y_t | f_t(x_t))$. Under the Margin algorithm this drives $p_t$ close to 1 for nearly every example (Eq. 5), meaning the Bernoulli gate almost never blocks an update. Similarly, in the Auxiliary variant the logit loss $\ell(\cdot)$ stays large, so the ratio $p_t = \ell / h$ stays close to 1 regardless of $\gamma$. In this regime the conservative algorithms degenerate into essentially the same behaviour as NC: they update on almost every example and produce a classifier that is just as dense, while incurring the extra overhead of computing the sampling probability each round. Therefore, on noisy or highly overlapping datasets, the sparsity advantage vanishes and the conservative methods offer no practical benefit — and may even be marginally slower than NC due to the per-step probability computation. The paper's own experiments hint at this: on the more difficult `a9a` dataset (Figure 4b), the sparsity gains from increasing $\gamma$ are noticeably smaller than on the easier `mushrooms` dataset.